# Results Overview — `uni_subject` Annotation (Education)

Aggregates results from `results/register.csv` for the `annotation / uni_subject` task.

> Run all cells top-to-bottom to regenerate after new results are submitted.

## 1. Imports & config

In [ ]:
import json as _json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np

REGISTER = Path("/home/tom/projects/corex_eval/results/register.csv")
TASK     = "annotation"
VAR      = "uni_subject"

FAMILY_ORDER  = ["Encoder", "OSS LLM", "Closed"]
FAMILY_COLORS = {
    "Encoder": "#4C72B0",
    "OSS LLM": "#55A868",
    "Closed":  "#C44E52",
}

## 2. Model catalogue

In [ ]:
# (family, display_name, experiment_folder)
CATALOGUE = [
    ("Encoder", "mBERT",           "bert_multilingual_uni_subject"),
    ("Encoder", "XLM-RoBERTa",     "xlmroberta_uni_subject"),
    ("OSS LLM", "Llama 3.3 70B",   "lmstudio_llama33_70b_uni_subject"),
    ("OSS LLM", "Qwen3 80B",       "lmstudio_qwen3_80b_uni_subject"),
    ("Closed",  "Claude Opus 4.6", "claude_opus46_uni_subject"),
    ("Closed",  "GPT-5.3",         "gpt53_uni_subject"),
]
print(f"{len(CATALOGUE)} experiments in catalogue")

## 3. Load register

In [ ]:
reg = pd.read_csv(REGISTER)
reg = reg[(reg["task"] == TASK) & (reg["variable"] == VAR)].copy()
print(f"Total rows after task/variable filter: {len(reg)}")

def _folder(path):
    parts = str(path).replace("\\", "/").split("/")
    return parts[-2] if len(parts) >= 2 else str(path)

reg["exp_folder"] = reg["experiment_path"].apply(_folder)
for col in ["accuracy", "macro_f1", "weighted_f1"]:
    reg[col] = pd.to_numeric(reg[col], errors="coerce")

reg_best = (
    reg[reg["accuracy"] > 0]
    .sort_values("accuracy", ascending=False)
    .drop_duplicates(subset="exp_folder", keep="first")
    .set_index("exp_folder")
)
print(f"Valid result rows: {len(reg_best)}")
print("Matched folders:", sorted(reg_best.index.tolist()))

## 4. Build results table

In [ ]:
records = []
for family, name, folder in CATALOGUE:
    if folder in reg_best.index:
        row = reg_best.loc[folder]
        records.append({
            "Family":      family,
            "Model":       name,
            "Accuracy":    round(float(row["accuracy"]),    4),
            "Macro F1":    round(float(row["macro_f1"]),    4) if pd.notna(row["macro_f1"])    else None,
            "Weighted F1": round(float(row["weighted_f1"]), 4) if pd.notna(row["weighted_f1"]) else None,
        })
    else:
        records.append({"Family": family, "Model": name,
                        "Accuracy": None, "Macro F1": None, "Weighted F1": None})

df = pd.DataFrame(records)
df["Family"] = pd.Categorical(df["Family"], categories=FAMILY_ORDER, ordered=True)
df = df.sort_values(["Family", "Model"]).reset_index(drop=True)

## 5. Results table

In [ ]:
def fmt_table(df):
    d = df.copy()
    for col in ["Accuracy", "Macro F1", "Weighted F1"]:
        d[col] = d[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "TBD")
    return d.style.hide(axis="index")

fmt_table(df)

In [ ]:
def to_latex(df, caption="uni\\_subject annotation", label="uni_subject"):
    d = df.copy()
    for col in ["Accuracy", "Macro F1", "Weighted F1"]:
        d[col] = d[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "--")
    rows, prev_fam = [], None
    for _, r in d.iterrows():
        if prev_fam is not None and r["Family"] != prev_fam:
            rows.append(r"\midrule")
        prev_fam = r["Family"]
        rows.append(f"{r['Family']} & {r['Model']} & {r['Accuracy']} & {r['Macro F1']} & {r['Weighted F1']} \\\\")
    body = "\n".join(rows)
    return (
        "\\begin{table}[ht]\n\\centering\n\\small\n"
        "\\begin{tabular}{llccc}\n\\toprule\n"
        "Family & Model & Accuracy & Macro F1 & Weighted F1 \\\\\n\\midrule\n"
        + body + "\n\\bottomrule\n\\end{tabular}\n"
        f"\\caption{{{caption}}}\n\\label{{tab:{label}}}\n\\end{{table}}"
    )

print(to_latex(df))

## 6. Bar chart

In [ ]:
def _rgba(hex_color, alpha):
    r, g, b = mcolors.to_rgb(hex_color)
    return (r, g, b, alpha)

has = df[df["Accuracy"].notna()].copy()
tbd = df[df["Accuracy"].isna()]["Model"].tolist()

x, w = np.arange(len(has)), 0.26
bar_colors = [FAMILY_COLORS[f] for f in has["Family"]]

fig, ax = plt.subplots(figsize=(max(8, len(has) * 1.1), 5))
for (metric, alpha), offset in zip([("Accuracy",1.0),("Macro F1",0.6),("Weighted F1",0.35)], [-w,0,w]):
    vals   = has[metric].values.astype(float)
    colors = [_rgba(c, alpha) for c in bar_colors]
    bars   = ax.bar(x + offset, vals, w, color=colors, edgecolor="white", linewidth=0.5, label=metric)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([f"{r.Model}\n({r.Family})" for _, r in has.iterrows()], fontsize=9, rotation=30, ha="right")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("uni_subject annotation — accuracy by model", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

family_patches = [mpatches.Patch(color=FAMILY_COLORS[f], label=f) for f in FAMILY_ORDER]
metric_patches = [mpatches.Patch(color="grey", alpha=a, label=m)
                  for m, a in [("Accuracy",1.0),("Macro F1",0.6),("Weighted F1",0.35)]]
ax.legend(handles=family_patches + metric_patches, fontsize=8, ncol=2, loc="upper right")
if tbd:
    ax.text(0.01, 0.02, "TBD: " + ", ".join(tbd), transform=ax.transAxes, fontsize=8, color="grey")
plt.tight_layout()
plt.show()

## 7. Per-country accuracy

In [ ]:
MODEL_FOR_MAP = "claude_opus46_uni_subject"  # change to any folder in reg_best

def get_per_country(folder):
    if folder not in reg_best.index:
        print(f"'{folder}' not found. Available: {sorted(reg_best.index.tolist())}")
        return {}
    mj = reg_best.loc[folder, "metrics_json"]
    if pd.isna(mj) or not mj: return {}
    try: return _json.loads(str(mj)).get("per_country", {})
    except: return {}

pc = get_per_country(MODEL_FOR_MAP)
if not pc:
    print("No per-country data found.")
else:
    pc_df = pd.DataFrame([
        {"country": k, "accuracy": v["accuracy"], "n": v["n_evaluated"]}
        for k, v in sorted(pc.items(), key=lambda x: x[1]["accuracy"], reverse=True)
    ])
    fig, ax = plt.subplots(figsize=(10, max(5, len(pc_df)*0.38)))
    colors = [mcolors.to_rgba("steelblue", 0.5 + 0.5*v) for v in pc_df["accuracy"]]
    bars   = ax.barh(pc_df["country"], pc_df["accuracy"], color=colors, edgecolor="white")
    for bar, (_, row) in zip(bars, pc_df.iterrows()):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f"{row['accuracy']:.3f}  (n={row['n']:.0f})", va="center", fontsize=8)
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Accuracy")
    ax.set_title(f"Per-country accuracy — {MODEL_FOR_MAP}", fontsize=12, fontweight="bold")
    ax.axvline(pc_df["accuracy"].mean(), color="tomato", linewidth=1.2, linestyle="--",
               label=f"Mean = {pc_df['accuracy'].mean():.3f}")
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()